# Topic 17F — Post-Cleaning Audit and Export

Cleaning is not finished when the code runs. It is finished when you verify the problems are resolved and save a clean output.

**Main lesson link:** [Topic 17 — Data Cleaning with pandas](topic17_data_cleaning.ipynb)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

filename = "data_cleaning_with_pandas/data/students_messy.csv"
possible_paths = [Path(filename), Path("data") / filename, Path("../data") / filename, Path("../../data") / filename]
data_path = next(path for path in possible_paths if path.exists())
df = pd.read_csv(data_path)
print("Loaded:", data_path)
print(df.shape)


---
## 1. A Compact Cleaning Pipeline

This cell applies a small version of the student cleaning workflow so we have a clean DataFrame to audit.


In [ ]:
clean = df.copy()
clean["full_name"] = clean["full_name"].str.strip()
clean = clean.drop_duplicates(subset="student_id", keep="first")
clean["region"] = clean["region"].str.strip().str.title().replace({
    "Gr. Accra": "Greater Accra",
    "G.Accra": "Greater Accra",
    "Asante": "Ashanti",
    "Eastern Region": "Eastern",
    "Western Region": "Western",
    "U/E": "Upper East",
})
clean["math_score"] = pd.to_numeric(clean["math_score"].astype(str).str.replace("%", "", regex=False), errors="coerce")
clean.loc[(clean["math_score"] < 0) | (clean["math_score"] > 100), "math_score"] = np.nan
for col in ["math_score", "english_score", "science_score"]:
    clean[col] = clean[col].fillna(clean[col].median())
clean["date_enrolled"] = pd.to_datetime(clean["date_enrolled"], dayfirst=True, errors="coerce")
clean["avg_score"] = clean[["math_score", "english_score", "science_score"]].mean(axis=1).round(1)
print(clean.head())


---
## 2. Final Audit Checks


In [ ]:
print("Rows:", len(clean))
print("Duplicate IDs:", clean["student_id"].duplicated().sum())
print("Missing values:")
print(clean.isna().sum())
print()
print("Score ranges:")
print(clean[["math_score", "english_score", "science_score", "avg_score"]].agg(["min", "max"]))
print()
print("Date dtype:", clean["date_enrolled"].dtype)


---
## 3. Create Data Quality Flags

Sometimes you should preserve a note about records that were fixed or need review.


In [ ]:
clean["quality_flag"] = "OK"
clean.loc[clean["date_enrolled"].isna(), "quality_flag"] = "CHECK_DATE"

print(clean["quality_flag"].value_counts())


---
## 4. Export the Clean Dataset


In [ ]:
output_path = "students_clean_practice.csv"
clean.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Rows:", len(clean))
print("Columns:", len(clean.columns))


### Useful Resources

- [pandas API: `DataFrame.to_csv`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
- [pandas API: `DataFrame.isna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html)
- [pandas getting started: Read and write tabular data](https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html)


---
## Quick Reflection

1. Why should you audit after cleaning?
2. Why might a `quality_flag` column be useful?
3. Why use `index=False` when exporting a clean CSV?
